In [25]:
# Import the libraries we will need
import pandas as pd
import numpy as np
import sklearn
import imblearn
import scipy

In [23]:
# Call specific packages from the libraries above
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

# Random Forest Classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV

# Imbalanced classes
from imblearn.over_sampling import SMOTE

In [ ]:
# Define our data ingestion function
def ingest_and_prep_data(
    bank_dataset: str = 'bank_data/bank.csv'
    ) -> tuple[scipy.sparse._csr.csr_matrix, pd.DataFrame, pd.Series, pd.Series]:
    """
    A function to ingest a dataset, then define training and test data.

    Parameters:
    - bank_dataset: str: the path to the dataset to be ingested.
    
    Returns:
    – X_train: pd.DataFrame: the training data features.
    – X_test: pd.DataFrame: the test data features.
    – y_train: pd.DataFrame: the training data target.
    – y_test: pd.DataFrame: the test data target.
    """
    df = pd.read_csv(bank_dataset, delimiter=';', decimal=',')
    
    # Assume there was some EDA and feature analysis to select below
    feature_cols = ['job', 'marital', 'education', 'contact', 'housing', 'loan', 'default', 'day']

    # Features and target
    X = df[feature_cols].copy()
    y = df['y'].apply(lambda x: 1 if x == 'yes' else 0).copy()

    # Define training and test data
    X_train, X_test, y_train, y_test = train_test_split(X, y, 
                            test_size=0.2, 
                            random_state=42)

    # Feature engineering
    enc = OneHotEncoder(handle_unknown='ignore')
    X_train = enc.fit_transform(X_train)
    
    return X_train, X_test, y_train, y_test


In [27]:

# Define our rebalance classes function
def rebalance_classes(X: pd.DataFrame, y: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    sm = SMOTE()
    X_balanced, y_balanced = sm.fit_resample(X, y)
    return X_balanced, y_balanced

In [28]:
def get_hyperparam_grid() -> dict:
    # Hyperparameter optimisation
    n_estimators = [int(x) for x in np.linspace(start=200, stop=2000, num=10)]
    # Number of features to consider at every split
    max_features = ['log2', 'sqrt'] #['auto', 'sqrt'] #TODO: auto throws some errors, remove from book example?
    # Maximum number of levels in tree
    max_depth = [int(x) for x in np.linspace(10, 110, num=11)]
    max_depth.append(None)
    # Minimum number of samples required to split a node
    min_samples_split = [2, 5, 10]
    # Minimum number of samples required at each leaf node
    min_samples_leaf = [1, 2, 4]
    # Method of selecting samples for training each tree
    bootstrap = [True, False]  # Create the random grid
    random_grid = {'n_estimators': n_estimators,
                   'max_features': max_features,
                   'max_depth': max_depth,
                   'min_samples_split': min_samples_split,
                   'min_samples_leaf': min_samples_leaf,
                   'bootstrap': bootstrap}
    return random_grid

In [29]:
def get_randomised_rf_cv(random_grid: dict) -> sklearn.model_selection._search.RandomizedSearchCV:
    # Use the random grid to search for best hyperparameters
    # First create the base model to tune
    rf = RandomForestClassifier()
    # Random search of parameters, using 3 fold cross validation,
    # search across 100 different combinations, and use all available cores
    rf_random = RandomizedSearchCV(
        estimator=rf,
        param_distributions=random_grid,
        n_iter=100,
        cv=3,
        verbose=2,
        random_state=42,
        n_jobs=-1,
        scoring='f1'
    )
    return rf_random

In [30]:
if __name__ == "__main__":
    X_train, X_test, y_train, y_test = ingest_and_prep_data()
    
    X_balanced, y_balanced = rebalance_classes(X_train, y_train)
    
    rf_random = get_randomised_rf_cv(
        random_grid=get_hyperparam_grid()
        )
    
    rf_random.fit(X_balanced, y_balanced)

Fitting 3 folds for each of 100 candidates, totalling 300 fits


/Users/melissa/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
Python(27221) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(27222) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(27223) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(27224) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(27225) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(27226) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(27227) MallocStackLogging: can't turn off malloc stack logging because it was not e

[CV] END bootstrap=True, max_depth=30, max_features=sqrt, min_samples_leaf=1, min_samples_split=5, n_estimators=400; total time=   2.6s
[CV] END bootstrap=True, max_depth=30, max_features=sqrt, min_samples_leaf=1, min_samples_split=5, n_estimators=400; total time=   2.8s
[CV] END bootstrap=True, max_depth=30, max_features=sqrt, min_samples_leaf=1, min_samples_split=5, n_estimators=400; total time=   3.1s
[CV] END bootstrap=False, max_depth=10, max_features=sqrt, min_samples_leaf=2, min_samples_split=5, n_estimators=1200; total time=   4.3s
[CV] END bootstrap=False, max_depth=10, max_features=sqrt, min_samples_leaf=2, min_samples_split=5, n_estimators=1200; total time=   4.6s
[CV] END bootstrap=True, max_depth=10, max_features=sqrt, min_samples_leaf=1, min_samples_split=5, n_estimators=2000; total time=   6.2s
[CV] END bootstrap=False, max_depth=10, max_features=sqrt, min_samples_leaf=2, min_samples_split=5, n_estimators=1200; total time=   3.5s
[CV] END bootstrap=True, max_depth=10, ma